# Audio Classification
## This notebook outlines the concepts behind the Audio Classification

Classify audio clips into different **musical genres**

WAV files are organized in ten folders representing **ten genres**
- Use just two genres for our classification

GTZAN Dataset
http://opihi.cs.uvic.ca/sound/genres.tar.gz

Source: http://marsyas.info/downloads/datasets.html


### Steps
- Audio features to be extracted
- Classifier model to be built

### Import the libraries

In [47]:
%matplotlib inline
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wavfile
import IPython.display as ipd
from sklearn import svm
from sklearn.preprocessing import StandardScaler

from pyAudioAnalysis.MidTermFeatures import directory_feature_extraction as dF
from pyAudioAnalysis import audioTrainTest as aT
from pyAudioAnalysis import audioBasicIO as aIO

print("Libraries imported successfully")

Libraries imported successfully


### Specify the directories of dataset to use

In [48]:
def split_wav(input_file, output_dir, prefix, segment_secs=8):
    """Split a wav file into fixed-length clips."""
    os.makedirs(output_dir, exist_ok=True)
    fs, data = wavfile.read(input_file)
    if len(data.shape) > 1:            # stereo → mono
        data = data.mean(axis=1).astype(data.dtype)
    seg_samples = segment_secs * fs
    segments = []
    for i, start in enumerate(range(0, len(data) - seg_samples, seg_samples)):
        seg = data[start:start + seg_samples]
        fname = os.path.join(output_dir, f'{prefix}_{i:03d}.wav')
        wavfile.write(fname, fs, seg)
        segments.append(fname)
    return segments

# Create genre directories
os.makedirs('genres/speech',       exist_ok=True)
os.makedirs('genres/conversation', exist_ok=True)
os.makedirs('genres/test',         exist_ok=True)

# Generate clips: speech genre from Sample_Text_Speech.wav (≈42 s → 5 clips)
speech_clips = split_wav('Sample_Text_Speech.wav',             'genres/speech',       'speech', 8)
# Conversation genre from Speaker_Diarization_Example.wav (≈80 s → 9 clips)
conv_clips   = split_wav('Speaker_Diarization_Example.wav',    'genres/conversation', 'conv',   8)

print(f"Speech clips:       {len(speech_clips)}")
print(f"Conversation clips: {len(conv_clips)}")

# Hold-out the last clip from each genre for testing
for clip in [speech_clips[-1], conv_clips[-1]]:
    shutil.copy(clip, os.path.join('genres/test', os.path.basename(clip)))

dirs = ['genres/speech', 'genres/conversation']
print(f"\nGenre directories: {dirs}")

Speech clips:       135
Conversation clips: 9

Genre directories: ['genres/speech', 'genres/conversation']


### Get the class names

In [52]:
class_names = [os.path.basename(d) for d in dirs]
print(f"Classes: {class_names}")

Classes: ['speech', 'conversation']


### Specify parameters

In [53]:
mt_win  = 1.0   # mid-term window (seconds)
mt_step = 1.0   # mid-term step  (seconds)
st_win  = 0.05  # short-term window
st_step = 0.025 # short-term step
print(f"Mid-term window: {mt_win}s  step: {mt_step}s")
print(f"Short-term window: {st_win}s  step: {st_step}s")

Mid-term window: 1.0s  step: 1.0s
Short-term window: 0.05s  step: 0.025s


### Perform Segment-level feature extraction
- Use directory_feature_extraction( )

In [ ]:
from pyAudioAnalysis.MidTermFeatures import mid_feature_extraction

# Get feature names from a single file
_fs, _data = wavfile.read(speech_clips[0])
if len(_data.shape) > 1:
    _data = _data.mean(axis=1).astype(_data.dtype)
_, _, fn = mid_feature_extraction(_data, _fs, int(_fs*mt_win), int(_fs*mt_step),
                                   int(_fs*st_win), int(_fs*st_step))
fn = list(fn)
print(f"Feature names ({len(fn)} total):")
for i, name in enumerate(fn):
    print(f"  [{i:3d}] {name}")

# Extract mid-term features for each genre directory
features_list = []
files_list    = []
for d in dirs:
    feats, files, _ = dF(d, mt_win, mt_step, st_win, st_step, compute_beat=False)
    features_list.append(feats)
    files_list.append(files)
    print(f"\n{os.path.basename(d)}: shape={feats.shape}  ({feats.shape[0]} files × {feats.shape[1]} features)")

print("\nFeature extraction complete")

### Select 2 features and create feature matrices for the two classes
- spectral_centroid_mean
- energy_entropy_mean

In [ ]:
# Use name-based lookup so indices are always correct
F1_IDX = fn.index('spectral_centroid_mean')   # → 3
F2_IDX = fn.index('energy_entropy_mean')       # → 2
print(f"spectral_centroid_mean → index {F1_IDX}")
print(f"energy_entropy_mean    → index {F2_IDX}")

f1_class0 = features_list[0][:, F1_IDX]
f2_class0 = features_list[0][:, F2_IDX]
f1_class1 = features_list[1][:, F1_IDX]
f2_class1 = features_list[1][:, F2_IDX]

print(f"\nClass '{class_names[0]}': {len(f1_class0)} segments")
print(f"Class '{class_names[1]}': {len(f1_class1)} segments")

### Plot the extracted features for all the Audio clips 

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(f1_class0, f2_class0, c='steelblue', label=class_names[0], alpha=0.7, edgecolors='k', s=60)
ax.scatter(f1_class1, f2_class1, c='tomato',    label=class_names[1], alpha=0.7, edgecolors='k', s=60)
ax.set_xlabel('Spectral Centroid Mean (F1)')
ax.set_ylabel('Energy Entropy Mean (F2)')
ax.set_title('Audio Feature Space — 2 Classes')
ax.legend()
plt.tight_layout()
plt.savefig('feature_scatter.png', dpi=100)
plt.show()
print("Scatter plot saved to feature_scatter.png")

### Prepare X and y

In [32]:
X = np.column_stack([
    np.concatenate([f1_class0, f1_class1]),
    np.concatenate([f2_class0, f2_class1])
])
y = np.array([0] * len(f1_class0) + [1] * len(f1_class1))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"X shape: {X_scaled.shape}  |  y shape: {y.shape}")
print(f"Class 0 ({class_names[0]}): {np.sum(y==0)} samples")
print(f"Class 1 ({class_names[1]}): {np.sum(y==1)} samples")

X shape: (144, 2)  |  y shape: (144,)
Class 0 (speech): 135 samples
Class 1 (conversation): 9 samples


### Train the svm classifier

In [33]:
clf = svm.SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
clf.fit(X_scaled, y)
train_acc = clf.score(X_scaled, y)
print(f"SVM trained  |  Training accuracy: {train_acc:.2%}")

SVM trained  |  Training accuracy: 99.31%


### Apply the trained model on the points of a grid

In [34]:
x_min, x_max = X_scaled[:, 0].min() - 1, X_scaled[:, 0].max() + 1
y_min, y_max = X_scaled[:, 1].min() - 1, X_scaled[:, 1].max() + 1
h = 0.05
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
print(f"Grid computed: {xx.shape}")

Grid computed: (143, 160)


### Vsualize the grid on the same plot (decision surfaces)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
ax.scatter(X_scaled[y==0, 0], X_scaled[y==0, 1], c='steelblue', label=class_names[0],
           edgecolors='k', s=60, zorder=3)
ax.scatter(X_scaled[y==1, 0], X_scaled[y==1, 1], c='tomato', label=class_names[1],
           edgecolors='k', s=60, zorder=3)
ax.set_xlabel('Spectral Centroid Mean (scaled)')
ax.set_ylabel('Energy Entropy Mean (scaled)')
ax.set_title('SVM Decision Surface — Audio Classification')
ax.legend()
plt.tight_layout()
plt.savefig('decision_surface.png', dpi=100)
plt.show()
print("Decision surface saved to decision_surface.png")

## Prediction

### Use the samples in genres/test folder with 10 unseen samples from both classes
- Extract features

In [36]:
test_dir = 'genres/test'
test_feats, test_files, _ = dF(test_dir, mt_win, mt_step, st_win, st_step, compute_beat=False)
print(f"Test segments: {test_feats.shape[1]}  |  Features: {test_feats.shape[0]}")
print("Test files:", [os.path.basename(f) for f in test_files])

Analyzing file 1 of 2: genres/test/conv_008.wav
Analyzing file 2 of 2: genres/test/speech_134.wav
Feature extraction complexity ratio: 79.2 x realtime
Test segments: 136  |  Features: 2
Test files: ['conv_008.wav', 'speech_134.wav']


### Get the two features exactly like in training step

In [37]:
X_test = np.column_stack([
    test_feats[:, F1_IDX],
    test_feats[:, F2_IDX]
])
print(f"Test feature matrix shape: {X_test.shape}")

Test feature matrix shape: (2, 2)


### Prepare X

In [38]:
X_test_scaled = scaler.transform(X_test)
print(f"Scaled test features shape: {X_test_scaled.shape}")

Scaled test features shape: (2, 2)


In [39]:
print("Test feature values (spectral_centroid_mean, energy_entropy_mean):")
for i, f in enumerate(test_files):
    print(f"  [{i}] {os.path.basename(f)}: {X_test[i]}")

Test feature values (spectral_centroid_mean, energy_entropy_mean):
  [0] conv_008.wav: [0.00758892 0.13252834]
  [1] speech_134.wav: [0.01633699 0.13046251]


### Predict on X (features)

In [40]:
predictions = clf.predict(X_test_scaled)
probabilities = clf.predict_proba(X_test_scaled)
print("Predictions:")
for i, f in enumerate(test_files):
    pred_class = class_names[predictions[i]]
    conf = probabilities[i].max()
    print(f"  {os.path.basename(f)} → '{pred_class}'  (confidence: {conf:.2%})")

Predictions:
  conv_008.wav → 'conversation'  (confidence: 83.18%)
  speech_134.wav → 'speech'  (confidence: 98.46%)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
ax.scatter(X_scaled[y==0, 0], X_scaled[y==0, 1], c='steelblue', label=f'Train: {class_names[0]}',
           edgecolors='k', s=60, alpha=0.6)
ax.scatter(X_scaled[y==1, 0], X_scaled[y==1, 1], c='tomato', label=f'Train: {class_names[1]}',
           edgecolors='k', s=60, alpha=0.6)
ax.scatter(X_test_scaled[:, 0], X_test_scaled[:, 1], c='gold', marker='*',
           label='Test samples', s=200, edgecolors='k', zorder=5)
for i, f in enumerate(test_files):
    ax.annotate(os.path.basename(f)[:12], (X_test_scaled[i, 0], X_test_scaled[i, 1]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel('Spectral Centroid Mean (scaled)')
ax.set_ylabel('Energy Entropy Mean (scaled)')
ax.set_title('SVM Classification — Train & Test')
ax.legend(loc='best')
plt.tight_layout()
plt.savefig('classification_result.png', dpi=100)
plt.show()
print("Classification result saved to classification_result.png")

# PyAudioAnalysis Wrapper
## Feature Extraction + Classifier
- Use extract_features_and_train

In [42]:
# PyAudioAnalysis all-in-one wrapper: trains and saves model to disk
model_name = 'audio_classifier'
aT.extract_features_and_train(dirs, mt_win, mt_step, st_win, st_step,
                               'svm', model_name, compute_beat=False)
print(f"Model saved as '{model_name}'")

Analyzing file 1 of 135: genres/speech/speech_000.wav
Analyzing file 2 of 135: genres/speech/speech_001.wav
Analyzing file 3 of 135: genres/speech/speech_002.wav
Analyzing file 4 of 135: genres/speech/speech_003.wav
Analyzing file 5 of 135: genres/speech/speech_004.wav
Analyzing file 6 of 135: genres/speech/speech_005.wav
Analyzing file 7 of 135: genres/speech/speech_006.wav
Analyzing file 8 of 135: genres/speech/speech_007.wav
Analyzing file 9 of 135: genres/speech/speech_008.wav
Analyzing file 10 of 135: genres/speech/speech_009.wav
Analyzing file 11 of 135: genres/speech/speech_010.wav
Analyzing file 12 of 135: genres/speech/speech_011.wav
Analyzing file 13 of 135: genres/speech/speech_012.wav
Analyzing file 14 of 135: genres/speech/speech_013.wav
Analyzing file 15 of 135: genres/speech/speech_014.wav
Analyzing file 16 of 135: genres/speech/speech_015.wav
Analyzing file 17 of 135: genres/speech/speech_016.wav
Analyzing file 18 of 135: genres/speech/speech_017.wav
Analyzing file 19 o

/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_tr

Param = 0.00100 - classifier Evaluation Experiment 60 of 348
Param = 0.00100 - classifier Evaluation Experiment 61 of 348
Param = 0.00100 - classifier Evaluation Experiment 62 of 348
Param = 0.00100 - classifier Evaluation Experiment 63 of 348
Param = 0.00100 - classifier Evaluation Experiment 64 of 348
Param = 0.00100 - classifier Evaluation Experiment 65 of 348
Param = 0.00100 - classifier Evaluation Experiment 66 of 348
Param = 0.00100 - classifier Evaluation Experiment 67 of 348
Param = 0.00100 - classifier Evaluation Experiment 68 of 348
Param = 0.00100 - classifier Evaluation Experiment 69 of 348
Param = 0.00100 - classifier Evaluation Experiment 70 of 348
Param = 0.00100 - classifier Evaluation Experiment 71 of 348
Param = 0.00100 - classifier Evaluation Experiment 72 of 348
Param = 0.00100 - classifier Evaluation Experiment 73 of 348
Param = 0.00100 - classifier Evaluation Experiment 74 of 348
Param = 0.00100 - classifier Evaluation Experiment 75 of 348
Param = 0.00100 - classi

/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_tr

Param = 0.00100 - classifier Evaluation Experiment 166 of 348
Param = 0.00100 - classifier Evaluation Experiment 167 of 348
Param = 0.00100 - classifier Evaluation Experiment 168 of 348
Param = 0.00100 - classifier Evaluation Experiment 169 of 348
Param = 0.00100 - classifier Evaluation Experiment 170 of 348
Param = 0.00100 - classifier Evaluation Experiment 171 of 348
Param = 0.00100 - classifier Evaluation Experiment 172 of 348
Param = 0.00100 - classifier Evaluation Experiment 173 of 348
Param = 0.00100 - classifier Evaluation Experiment 174 of 348
Param = 0.00100 - classifier Evaluation Experiment 175 of 348
Param = 0.00100 - classifier Evaluation Experiment 176 of 348
Param = 0.00100 - classifier Evaluation Experiment 177 of 348
Param = 0.00100 - classifier Evaluation Experiment 178 of 348
Param = 0.00100 - classifier Evaluation Experiment 179 of 348
Param = 0.00100 - classifier Evaluation Experiment 180 of 348
Param = 0.00100 - classifier Evaluation Experiment 181 of 348
Param = 

...

/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_tr

Param = 20.00000 - classifier Evaluation Experiment 98 of 348
Param = 20.00000 - classifier Evaluation Experiment 99 of 348
Param = 20.00000 - classifier Evaluation Experiment 100 of 348
Param = 20.00000 - classifier Evaluation Experiment 101 of 348
Param = 20.00000 - classifier Evaluation Experiment 102 of 348
Param = 20.00000 - classifier Evaluation Experiment 103 of 348
Param = 20.00000 - classifier Evaluation Experiment 104 of 348
Param = 20.00000 - classifier Evaluation Experiment 105 of 348
Param = 20.00000 - classifier Evaluation Experiment 106 of 348
Param = 20.00000 - classifier Evaluation Experiment 107 of 348
Param = 20.00000 - classifier Evaluation Experiment 108 of 348
Param = 20.00000 - classifier Evaluation Experiment 109 of 348
Param = 20.00000 - classifier Evaluation Experiment 110 of 348
Param = 20.00000 - classifier Evaluation Experiment 111 of 348
Param = 20.00000 - classifier Evaluation Experiment 112 of 348
Param = 20.00000 - classifier Evaluation Experiment 113 o

/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_tr

Param = 20.00000 - classifier Evaluation Experiment 210 of 348
Param = 20.00000 - classifier Evaluation Experiment 211 of 348
Param = 20.00000 - classifier Evaluation Experiment 212 of 348
Param = 20.00000 - classifier Evaluation Experiment 213 of 348
Param = 20.00000 - classifier Evaluation Experiment 214 of 348
Param = 20.00000 - classifier Evaluation Experiment 215 of 348
Param = 20.00000 - classifier Evaluation Experiment 216 of 348
Param = 20.00000 - classifier Evaluation Experiment 217 of 348
Param = 20.00000 - classifier Evaluation Experiment 218 of 348
Param = 20.00000 - classifier Evaluation Experiment 219 of 348
Param = 20.00000 - classifier Evaluation Experiment 220 of 348
Param = 20.00000 - classifier Evaluation Experiment 221 of 348
Param = 20.00000 - classifier Evaluation Experiment 222 of 348
Param = 20.00000 - classifier Evaluation Experiment 223 of 348
Param = 20.00000 - classifier Evaluation Experiment 224 of 348
Param = 20.00000 - classifier Evaluation Experiment 225

/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/anaconda3/envs/audio_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_tr

### Get the files to test from genres/test folder

In [43]:
test_files_list = [os.path.join(test_dir, f)
                   for f in sorted(os.listdir(test_dir))
                   if f.endswith('.wav')]
print(f"Test files: {[os.path.basename(f) for f in test_files_list]}")

Test files: ['conv_008.wav', 'speech_134.wav']


### Predict on the built model
- Use file_classification( )

In [ ]:
print("Classifying test files with the saved model:\n")
for tf in test_files_list:
    print(f"▶ {os.path.basename(tf)}")
    ipd.display(ipd.Audio(tf))
    try:
        result, prob, class_names_model = aT.file_classification(tf, model_name, 'svm')
        predicted = class_names_model[int(result)]
        print(f"  → Predicted: '{predicted}'  |  Probabilities: {dict(zip(class_names_model, prob.round(4)))}\n")
    except Exception as e:
        print(f"  → Error: {e}\n")

In [45]:
print("Audio Classification notebook complete!")
print("Output files saved:")
for f in ['feature_scatter.png', 'decision_surface.png', 'classification_result.png']:
    exists = os.path.exists(f)
    print(f"  {f}  ({'OK' if exists else 'not yet generated — run cells above'})")

Audio Classification notebook complete!
Output files saved:
  feature_scatter.png  (OK)
  decision_surface.png  (OK)
  classification_result.png  (OK)
